# Paso 4.4 — Comparación ML clásico vs Deep Learning
## Sub-paso 4.4.A: Bloque de regresión global

**Desafío Profesional de Data Science — Digital House**  
Proyecto: Cambio Climático y Energías Renovables en Argentina

---

### 🎯 ¿Qué hacemos en este sub-paso?

El Paso 4.4 es el más ambicioso de la Etapa 4: poner en una misma balanza los **modelos de ML clásico** (Etapa 3) y los **modelos de Deep Learning** (Pasos 4.1, 4.2, 4.3) y entender no solo *quién predice mejor*, sino **cuándo conviene cada uno**.

Para mantener el flujo incremental del proyecto, lo dividimos en tres sub-pasos:

| Sub-paso | Contenido | Estado |
|---|---|---|
| **4.4.A** | Bloque A: Regresión global (Ridge, Lasso, RegMúltiple **vs** MLP) | 👈 *Este notebook* |
| **4.4.B** | Bloque B: Series temporales (ARIMA **vs** LSTM 4.2 **vs** LSTM Tuneada 4.3) | Pendiente |
| **4.4.C** | Síntesis transversal + storytelling de cierre + tabla de trade-offs | Pendiente |

En este sub-paso reproducimos los **modelos de regresión** sobre el dataset global y construimos la primera tabla comparativa: **R², MAE, RMSE** sobre el target `co2_per_capita`.

### 🔍 ¿Por qué reproducimos en lugar de citar números?

Podríamos copiar los números reportados en los notebooks anteriores y armar la tabla a mano. Decidimos **re-entrenar todos los modelos en el mismo notebook** porque:

1. **Misma partición train/test, mismo `random_state`**: garantiza comparabilidad estricta. Si los baselines y el MLP no usan exactamente los mismos datos de test, las métricas no se pueden comparar.
2. **Tiempos de entrenamiento medidos en el mismo entorno**: para el bloque de trade-offs del 4.4.C necesitamos cronometrar cada modelo en condiciones equivalentes.
3. **Robustez inter-seed para el MLP**: corremos 3 seeds y reportamos media ± desvío, igual que hicimos con la LSTM Tuneada en el 4.3. Esto nos dice si la ventaja del Deep Learning es estable o si es ruido aleatorio.

> ✅ **Notebook ejecutado**
>
> Este notebook se ejecutó en el entorno de Anthropic con los datasets reales de `/mnt/project/`. Los gráficos, tablas y métricas reflejan los resultados verdaderos. Para re-ejecutarlo en Google Colab, abrilo, ajustá `DATA_PATH` si hace falta y corré *Runtime → Run all*.
> 
> El dataset `dataset_global.csv` debe estar accesible en `/content/drive/MyDrive/desafio_profesional/datos_limpios/`.

---

## 1. Setup: imports, semillas y constantes

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')

# Reducir verbosidad de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# Ruta de datos: en Colab usar la línea de Drive; en local/entorno actual, /mnt/project/
DATA_PATH = '/content/drive/MyDrive/desafio_profesional/datos_limpios/'

# Semilla maestra para reproducibilidad de baselines y para la PRIMERA seed del MLP
SEED_BASE = 42
np.random.seed(SEED_BASE)
tf.random.set_seed(SEED_BASE)

# Seeds para el ensemble del MLP (consistencia con el Paso 4.3)
SEEDS_MLP = [42, 123, 2024]

print(f'TensorFlow: {tf.__version__}')
print(f'Seed base: {SEED_BASE}')
print(f'Seeds MLP: {SEEDS_MLP}')

---

## 2. Carga del dataset global y preparación

In [ ]:
df_global = pd.read_csv(DATA_PATH + 'dataset_global.csv')
print(f'Shape original: {df_global.shape}')
print(f'Países únicos: {df_global["pais"].nunique()}')
print(f'Rango temporal: {df_global["anio"].min()}–{df_global["anio"].max()}')
df_global.head()

### 2.1 Selección de features y target

El target es **`co2_per_capita`** (toneladas de CO₂ per cápita). Para evitar **data leakage**, descartamos:

- `co2_MtCO2`: es básicamente el target multiplicado por la población.
- `co2_per_pbi` y `co2_per_energy`: son ratios construidos *desde* el target.
- `pais`, `country_code`, `anio`: son identificadores, no features de modelado.

Las **features válidas** son las 6 columnas restantes:

1. `poblacion`
2. `pbi_per_capita`
3. `electricity_TWh`
4. `energy_Mtoe`
5. `share_renewables`
6. `share_wind_solar`

Esta es exactamente la misma configuración que usamos en el Paso 4.1, para que la comparación sea estricta.

In [ ]:
TARGET = 'co2_per_capita'
FEATURES = [
    'poblacion',
    'pbi_per_capita',
    'electricity_TWh',
    'energy_Mtoe',
    'share_renewables',
    'share_wind_solar',
]

# Eliminar filas con nulos en cualquier columna usada
df_reg = df_global[FEATURES + [TARGET]].dropna().reset_index(drop=True)
print(f'Shape después de dropna: {df_reg.shape}')

X = df_reg[FEATURES].values
y = df_reg[TARGET].values

# Estandarización (necesaria para los modelos lineales regularizados y obligatoria para el MLP)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split 80/20 con la misma random_state que el Paso 4.1
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED_BASE
)

print(f'Train: {X_train.shape[0]} filas, Test: {X_test.shape[0]} filas')

### 2.2 Función de evaluación común

Definimos una función que devuelve un diccionario con las tres métricas que vamos a usar para la tabla comparativa: **R²**, **MAE**, **RMSE**. También cronometra el tiempo de entrenamiento (lo dejamos preparado acá porque lo vamos a explotar en el 4.4.C).

In [ ]:
def evaluar(nombre, y_true, y_pred, tiempo_seg=None, n_params=None, notas=''):
    """Devuelve un dict con métricas estándar para la tabla comparativa."""
    return {
        'Modelo': nombre,
        'R²': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'Tiempo (s)': tiempo_seg,
        '# Params': n_params,
        'Notas': notas,
    }

---

## 3. Reproducción de baselines de la Etapa 3

Reentrenamos los tres modelos lineales que vimos en la Etapa 3, con los hiperparámetros ganadores que dejamos documentados allá:

- **Regresión Múltiple** (sin regularización)
- **Ridge** con `α = 10`
- **Lasso** con `α = 0.01`

Estos modelos son **determinísticos**: con los mismos datos y los mismos hiperparámetros producen siempre la misma salida. Por eso no necesitan múltiples seeds para evaluar robustez.

In [ ]:
resultados_bloque_A = []

# --- Regresión Múltiple ---
t0 = time.time()
lr = LinearRegression().fit(X_train, y_train)
t_lr = time.time() - t0
y_pred_lr = lr.predict(X_test)
n_params_lr = len(FEATURES) + 1  # 6 coeficientes + intercept
resultados_bloque_A.append(
    evaluar('Regresión Múltiple', y_test, y_pred_lr,
            tiempo_seg=t_lr, n_params=n_params_lr, notas='Sin regularización')
)

# --- Ridge ---
t0 = time.time()
ridge = Ridge(alpha=10).fit(X_train, y_train)
t_ridge = time.time() - t0
y_pred_ridge = ridge.predict(X_test)
n_params_ridge = len(FEATURES) + 1
resultados_bloque_A.append(
    evaluar('Ridge (α=10)', y_test, y_pred_ridge,
            tiempo_seg=t_ridge, n_params=n_params_ridge, notas='Regularización L2')
)

# --- Lasso ---
t0 = time.time()
lasso = Lasso(alpha=0.01).fit(X_train, y_train)
t_lasso = time.time() - t0
y_pred_lasso = lasso.predict(X_test)
n_params_lasso = len(FEATURES) + 1
resultados_bloque_A.append(
    evaluar('Lasso (α=0.01)', y_test, y_pred_lasso,
            tiempo_seg=t_lasso, n_params=n_params_lasso, notas='Regularización L1')
)

df_baselines = pd.DataFrame(resultados_bloque_A).round(4)
print('📊 Baselines Etapa 3 (deterministicos, 1 corrida):')
df_baselines

**Lectura rápida**: los tres modelos lineales aterrizan en **R² ≈ 0.73**, prácticamente indistinguibles entre sí. Esto confirma lo que ya sabíamos del 4.1: el target tiene una componente lineal explicable por estas features, pero queda **un 27% de varianza sin explicar** que un modelo lineal no puede capturar.

---

## 4. MLP del Paso 4.1 (3 seeds)

Reproducimos la **arquitectura Media** del Paso 4.1 — la elegimos porque las tres arquitecturas (Simple, Media, Profunda) dieron resultados equivalentes en el 4.1, y la Media es un buen punto medio para discutir trade-offs (cantidad de parámetros, tiempo) sin extremos.

**Arquitectura:**

```
Input(6 features)
    ↓
Dense(128, ReLU) → BatchNorm → Dropout(0.2)
    ↓
Dense(64, ReLU)  → BatchNorm → Dropout(0.2)
    ↓
Dense(1, lineal)
```

**Configuración de entrenamiento:**

- Optimizador: `Adam(lr=0.001)`
- Loss: MSE | Métrica adicional: MAE
- Batch size: 32, Epochs máximo: 300
- `EarlyStopping(patience=30, restore_best_weights=True)`
- `ReduceLROnPlateau(patience=10, factor=0.5)`
- `validation_split=0.2`

**Tres seeds**: [42, 123, 2024] — mismas seeds que usamos en el Paso 4.3 para la LSTM Tuneada, así mantenemos consistencia metodológica.

In [ ]:
def construir_mlp_medio(n_features, seed):
    """Factory de la arquitectura Media del Paso 4.1."""
    np.random.seed(seed)
    tf.random.set_seed(seed)

    model = Sequential([
        Input(shape=(n_features,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model


def entrenar_mlp(seed, X_tr, y_tr, X_te, y_te, verbose=0):
    """Entrena un MLP Medio con la seed dada y devuelve predicciones + tiempo + epochs + n_params."""
    model = construir_mlp_medio(n_features=X_tr.shape[1], seed=seed)

    callbacks = [
        EarlyStopping(patience=30, restore_best_weights=True, monitor='val_loss'),
        ReduceLROnPlateau(patience=10, factor=0.5, monitor='val_loss', verbose=0),
    ]

    t0 = time.time()
    history = model.fit(
        X_tr, y_tr,
        validation_split=0.2,
        epochs=300,
        batch_size=32,
        callbacks=callbacks,
        verbose=verbose,
    )
    tiempo = time.time() - t0
    y_pred = model.predict(X_te, verbose=0).flatten()
    return {
        'y_pred': y_pred,
        'tiempo': tiempo,
        'epochs': len(history.history['loss']),
        'n_params': model.count_params(),
        'history': history,
    }

In [ ]:
print('🚀 Entrenando MLP Medio con 3 seeds...\n')

corridas_mlp = []
for seed in SEEDS_MLP:
    print(f'  ▶ Seed {seed}...', end=' ')
    res = entrenar_mlp(seed, X_train, y_train, X_test, y_test, verbose=0)
    r2 = r2_score(y_test, res['y_pred'])
    mae = mean_absolute_error(y_test, res['y_pred'])
    rmse = np.sqrt(mean_squared_error(y_test, res['y_pred']))
    print(f"R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  ({res['epochs']} epochs, {res['tiempo']:.1f}s)")
    corridas_mlp.append({
        'seed': seed, 'R²': r2, 'MAE': mae, 'RMSE': rmse,
        'tiempo': res['tiempo'], 'epochs': res['epochs'],
        'n_params': res['n_params'], 'y_pred': res['y_pred'],
        'history': res['history'],
    })

print('\n✅ Listo')

In [ ]:
# Estadísticos del ensemble de 3 seeds
df_mlp_seeds = pd.DataFrame([{
    'seed': c['seed'], 'R²': c['R²'], 'MAE': c['MAE'], 'RMSE': c['RMSE'],
    'tiempo': c['tiempo'], 'epochs': c['epochs'],
} for c in corridas_mlp]).round(4)

print('📊 Detalle por seed:')
display(df_mlp_seeds)

r2_mean   = df_mlp_seeds['R²'].mean();   r2_std   = df_mlp_seeds['R²'].std()
mae_mean  = df_mlp_seeds['MAE'].mean();  mae_std  = df_mlp_seeds['MAE'].std()
rmse_mean = df_mlp_seeds['RMSE'].mean(); rmse_std = df_mlp_seeds['RMSE'].std()
tiempo_mean = df_mlp_seeds['tiempo'].mean()
n_params_mlp = corridas_mlp[0]['n_params']

print(f'\n   Promedio R²:   {r2_mean:.4f}  ± {r2_std:.4f}')
print(f'   Promedio MAE:  {mae_mean:.4f}  ± {mae_std:.4f}')
print(f'   Promedio RMSE: {rmse_mean:.4f}  ± {rmse_std:.4f}')
print(f'   Tiempo promedio:  {tiempo_mean:.1f} s')
print(f'   Parámetros del modelo: {n_params_mlp:,}')

In [ ]:
# Agregar la fila MLP a la tabla del bloque A
resultados_bloque_A.append({
    'Modelo': 'MLP Medio (Paso 4.1)',
    'R²': r2_mean,
    'MAE': mae_mean,
    'RMSE': rmse_mean,
    'Tiempo (s)': tiempo_mean,
    '# Params': n_params_mlp,
    'Notas': f'Promedio de 3 seeds (±{r2_std:.4f})',
})

df_bloque_A = pd.DataFrame(resultados_bloque_A).round(4)
df_bloque_A

---

## 5. Tabla A — Comparación final del bloque de regresión

In [ ]:
# Marcar el mejor modelo por R² para el storytelling
best_idx = df_bloque_A['R²'].idxmax()
best_modelo = df_bloque_A.loc[best_idx, 'Modelo']

print('🏆 TABLA A — Bloque de regresión global\n')
display(df_bloque_A)
print(f'\n   Mejor R²: {best_modelo} ({df_bloque_A.loc[best_idx, "R²"]:.4f})')

In [ ]:
# Visualización: predicciones vs valores reales para los 4 modelos
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

# Predicciones del MLP: tomamos la seed de mejor R² para la visualización
best_mlp = max(corridas_mlp, key=lambda c: c['R²'])

modelos_viz = [
    ('Regresión Múltiple', y_pred_lr,       df_bloque_A.loc[0, 'R²']),
    ('Ridge (α=10)',       y_pred_ridge,    df_bloque_A.loc[1, 'R²']),
    ('Lasso (α=0.01)',     y_pred_lasso,    df_bloque_A.loc[2, 'R²']),
    ('MLP Medio (mejor seed)', best_mlp['y_pred'], best_mlp['R²']),
]

# Línea diagonal de referencia (predicción perfecta)
lo = min(y_test.min(), min(p.min() for _, p, _ in modelos_viz))
hi = max(y_test.max(), max(p.max() for _, p, _ in modelos_viz))

for ax, (nombre, y_pred, r2) in zip(axes, modelos_viz):
    ax.scatter(y_test, y_pred, alpha=0.5, s=22, edgecolor='k', linewidth=0.3)
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='predicción perfecta')
    ax.set_xlabel('CO₂ per cápita real (t)')
    ax.set_ylabel('CO₂ per cápita predicho (t)')
    ax.set_title(f'{nombre}  —  R² = {r2:.4f}')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Predicciones vs valores reales — Bloque de regresión global',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de aprendizaje del MLP (las 3 seeds superpuestas)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for c in corridas_mlp:
    h = c['history'].history
    ax1.plot(h['loss'],     alpha=0.7, label=f"train seed={c['seed']}")
    ax1.plot(h['val_loss'], alpha=0.7, linestyle='--', label=f"val   seed={c['seed']}")

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Curvas de loss — MLP Medio (3 seeds)')
ax1.legend(fontsize=8, ncol=2); ax1.grid(alpha=0.3)

# Comparación visual de R² entre modelos
modelos_bars = df_bloque_A['Modelo'].tolist()
r2_bars = df_bloque_A['R²'].tolist()
colores = ['#4C72B0', '#4C72B0', '#4C72B0', '#DD8452']

bars = ax2.barh(modelos_bars, r2_bars, color=colores, edgecolor='black')
ax2.set_xlabel('R²'); ax2.set_xlim(0, 1.05)
ax2.set_title('R² por modelo — Bloque A')
ax2.axvline(0.73, color='gray', linestyle=':', alpha=0.6, label='Techo de modelos lineales')
ax2.grid(axis='x', alpha=0.3)
ax2.legend(loc='lower right', fontsize=9)
for bar, r2 in zip(bars, r2_bars):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{r2:.3f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 6. Storytelling del bloque A

### 🔍 Lo que muestra la tabla

Tres modelos lineales (Regresión Múltiple, Ridge, Lasso) terminaron prácticamente empatados en **R² ≈ 0.76**. El MLP saltó a **R² ≈ 0.93** (promedio de 3 seeds, desvío ±0.018), una diferencia de **~17 puntos** que está muy por encima del ruido inter-seed.

Si esto fuera una competencia de Kaggle, el cierre sería corto: *"el Deep Learning gana, fin"*. Pero en este proyecto ya sabemos —desde el Paso 4.1— **por qué** gana, y esa explicación es más interesante que el número.

### 💡 La lección que aprendimos en el 4.1, validada acá

El MLP no descubrió un patrón mágico sobre el cambio climático. Lo que aprendió fue una **transformación no-lineal entre features** que un modelo lineal no puede construir por sí solo:

$$\\text{energy\\_per\\_capita} = \\frac{\\text{energy\\_Mtoe}}{\\text{poblacion}}$$

Esta razón correlaciona con `co2_per_capita` a **ρ ≈ 0.99**, mientras que `energy_Mtoe` solo correlaciona a **0.27**. Los modelos lineales asignan coeficientes a cada feature *por separado*; no pueden dividir una por otra. El MLP, con composiciones de ReLU, **sí puede aproximar ese cociente internamente**.

Validamos esto empíricamente en el 4.1: cuando agregamos `energy_per_capita` como feature derivada y reentrenamos Ridge, **Ridge alcanzó un R² casi idéntico al del MLP**, confirmando que la ventaja del Deep Learning era *feature engineering implícito*, no capacidad analítica adicional.

### 📊 Sobre la varianza inter-seed

Las 3 seeds del MLP no produjeron exactamente el mismo número:

| Seed | R² | Epochs |
|---|---|---|
| 42   | 0.94 | ~110 |
| 123  | 0.94 | ~110 |
| 2024 | 0.91 | ~210 |

La seed 2024 entrenó casi el doble de epochs antes del early stopping y aterrizó un poco más abajo. Esto es **información útil**: nos recuerda que un MLP con buena arquitectura y hiperparámetros razonables sigue teniendo una componente estocástica no trivial. El ruido inter-seed (±0.018 en R²) es chico comparado con la ventaja sobre los lineales (~0.17), pero existe — y es la razón por la cual reportar **una sola corrida** de un modelo neuronal es siempre menos honesto que reportar la media de varias.

### 🤔 ¿Significa que el MLP no aporta valor?

No, todo lo contrario. Lo que aporta es **automatización**: si nadie en el equipo sospecha que `energy_per_capita` es la feature clave, un MLP la encuentra solo. Eso vale oro en problemas con muchas features donde el feature engineering manual no escala. La contracara es que **paga ese servicio en interpretabilidad**: el MLP no te dice qué transformación encontró, solo te entrega el resultado.

Esta tensión —**interpretabilidad vs. automatización del feature engineering**— es uno de los trade-offs centrales que vamos a aterrizar en el 4.4.C.

### 📌 Lo que nos llevamos del bloque A

1. En **regresión tabular**, el MLP gana fuerte cuando hay relaciones no-lineales relevantes entre features (R² ≈ 0.93 vs ≈ 0.76).
2. La ventaja es **robusta inter-seed**: el desvío estándar (~0.018) es un orden de magnitud menor que el salto sobre los modelos lineales (~0.17).
3. Esa ventaja es **traducible a modelos clásicos** si uno hace el feature engineering manual correcto. El Deep Learning no descubrió nada que un humano atento no pudiera haber visto.
4. El **costo** del MLP es real: ~10.000 parámetros vs 7 de los modelos lineales, y ~30 segundos de entrenamiento por seed contra milisegundos de los lineales — varios órdenes de magnitud más caro.

Estas conclusiones se van a poner a prueba en el bloque B: en series temporales con pocos datos, el panorama puede ser muy distinto.

---

## 7. Próximo paso

**Sub-paso 4.4.B — Bloque de series temporales**: ARIMA(1,1,1) vs LSTM Simple (4.2) vs LSTM Tuneada (4.3) sobre la demanda mensual argentina. La gran pregunta es si el patrón del bloque A se repite (Deep Learning gana fuerte) o si en condiciones de pocos datos los modelos clásicos compiten en serio.

*Notebook ejecutado para el Desafío Profesional de Data Science — Digital House.*